In [71]:
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt

# import torch
# device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
# device = 'cpu'
# print(f'torch device: {device}')

from dataset import CustomDataset, transform_data
from models import p3p

In [72]:
SOLO_NAME =  'poisson3r8_vis'
SCENE = 'SimpleOffice'
# SCENE = 'WP16'
DATA_DIR = f'../output/{SCENE}/{SOLO_NAME}'

In [93]:
ds = CustomDataset(root=DATA_DIR, scene=SCENE, transform=None)

In [120]:
dc = ds[70]
[k for k in dc.keys()]
dc['g1_camera_pose']

array([-11.3989372,   1.5      ,  -5.145088 ,   0.       ,   0.       ,
         0.       ,   1.       ])

In [121]:
pose = dc['g1_camera_pose']
t = pose[:3].reshape(-1, 1)
def q2R(q):
    q = q / np.linalg.norm(q)
    x, y, z, w = q
    R = np.array([
        [1 - 2*y**2 - 2*z**2, 2*x*y - 2*w*z, 2*x*z + 2*w*y],
        [2*x*y + 2*w*z, 1 - 2*x**2 - 2*z**2, 2*y*z - 2*w*x],
        [2*x*z - 2*w*y, 2*y*z + 2*w*x, 1 - 2*x**2 - 2*y**2]
    ])
    return R
q = pose[3:]
R = q2R(q)
print(t)
print(R)
print(q)

[[-11.3989372]
 [  1.5      ]
 [ -5.145088 ]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[0. 0. 0. 1.]


In [122]:
K = dc['g1_camera_intrinsics']
K

array([[ 1.73082447,  0.        ,  0.        ],
       [ 0.        ,  1.73082447,  0.        ],
       [ 0.        ,  0.        , -1.0006001 ]])

In [198]:
obj_w = dc['node_df'][['pos_x', 'pos_y', 'pos_z']]
idx = [1,2,3]
obj_w = obj_w.iloc[idx]
# obj_w = obj_w.where(obj_w['pos_x'] <  -12).dropna()
# idx = obj_w.index
obj_w.head()

,pos_x,pos_y,pos_z
1,-15.26076,2.894000,1.110
2,-12.26076,2.894000,1.110
3,-11.75000,0.005001,-2.011


In [200]:
dc['node_df'][['bbox_x0', 'bbox_y0', 'bbox_cx', 'bbox_cy', 'bbox_w', 'bbox_h']].iloc[idx].head()

,bbox_x0,bbox_y0,bbox_cx,bbox_cy,bbox_w,bbox_h
1,0.0,98.0,27.5,140.0,55.0,84.0
2,121.0,98.0,155.5,155.0,69.0,114.0
3,97.0,194.0,129.0,250.5,64.0,113.0


In [149]:
# obj_w.where(obj_w['pos_x'] <  -12)[1:3]

In [208]:
obj_c = (R @ obj_w.T.values + t)
alpha, beta = 2*128/(3), 2*128/(3)
my_K = np.array([
    [-alpha, 0, 0],
    [0, beta, 256],
    [0, 0, 1]
])

# my_K
obj_px = (my_K @ obj_c)
obj_px = (obj_px / obj_px[2])
obj_px.T
# obj_c.T

array([[-563.79461886,  163.07645564,    1.        ],
       [-500.35114297,  163.07645564,    1.        ],
       [-276.04131339,  238.05350266,    1.        ]])